# Phase 3: Fine-Tuning (Financial Domain)
Using QLoRA to fine-tune an LLM on Financial QA data.

In [1]:
# CELL 1: Install Dependencies
!pip install trl peft bitsandbytes accelerate transformers datasets

In [2]:
# CELL 2: Login to HuggingFace
from huggingface_hub import login
from google.colab import userdata
login(userdata.get('HF_TOKEN'))

In [3]:
# CELL 3: Load Financial Dataset
from datasets import load_dataset

# Using gbharti/finance-alpaca - a well-known financial instruction dataset
dataset = load_dataset("gbharti/finance-alpaca")
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 68912
    })
})
{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers 

In [4]:
# CELL 4: Formatting Function
def format_prompt(sample):
    instruction = sample['instruction']
    input_text = sample.get('input', '')
    output = sample['output']

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    else:
        return f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

def add_text_column(sample):
    sample["text"] = format_prompt(sample)
    return sample

dataset = dataset.map(add_text_column)
print("Sample formatted prompt:")
print(dataset['train'][0]['text'])

Sample formatted prompt:
### Instruction:
For a car, what scams can be plotted with 0% financing vs rebate?

### Response:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 mon

In [7]:
# CELL 5: Load TinyLlama for Fine-Tuning
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

torch.cuda.empty_cache()
gc.collect()

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, peft_config)

# CRITICAL FIX: Force any bfloat16 layers to float16 (T4 doesn't support bfloat16)
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [8]:
# CELL 6: Start Training
from trl import SFTTrainer, SFTConfig

small_dataset = dataset['train'].select(range(500))

sft_config = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,        # FIXED: Disabled to avoid bfloat16 AMP conflict
    bf16=False,        # FIXED: T4 does not support bfloat16
    max_steps=50,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=small_dataset,
    args=sft_config,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.592931
20,2.553898
30,2.491490
40,2.457780
50,2.451230


TrainOutput(global_step=50, training_loss=2.5094660186767577, metrics={'train_runtime': 199.2896, 'train_samples_per_second': 2.007, 'train_steps_per_second': 0.251, 'total_flos': 709669915508736.0, 'train_loss': 2.5094660186767577, 'epoch': 0.8})

In [9]:
# CELL 7: Save the Fine-Tuned LoRA Adapter
model.save_pretrained("./financial_lora_adapter")
print("LoRA Adapter saved!")

LoRA Adapter saved!
